# Customer Churn Intelligence — Leakage-Safe Preprocessing

## Objective

Build and validate an sklearn `ColumnTransformer` that scales numeric features and one-hot encodes categorical features. **Fit on `X_train` only**; validation and test use `transform()` only.

**Stage:** Step 9 — Preprocessing only (no model training, SMOTE, or threshold tuning).

### Why these design choices?

- **`ColumnTransformer`** — applies the right transformation to each column group in one reproducible object shared by training and inference.
- **`OneHotEncoder`** — replaces manual category mapping, handles multi-level categoricals consistently, and `handle_unknown='ignore'` prevents crashes on unseen categories at inference time.
- **`StandardScaler`** — puts numeric features on comparable scales, which helps **Logistic Regression** converge and interpret regularized coefficients fairly across features.
- **Tree models** (Random Forest, XGBoost) do not strictly require scaling, but a **shared preprocessing architecture** keeps one pipeline for all models and simplifies Streamlit/FastAPI deployment later.
- **Fit on training data only** — prevents validation/test information from leaking into imputation, scaling, and encoding statistics (AGENTS.md rule 4).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_separation import CATEGORICAL_FEATURE_COLS, NUMERIC_FEATURE_COLS
from src.data_split import load_split_from_manifest
from src.preprocessing import (
    ORIGINAL_FEATURE_COUNT,
    build_preprocessor,
    fit_transform_train,
    run_preprocessing_pipeline,
    transform_splits,
    validate_transformed_splits,
)

## 1. Load Splits (Step 8)

In [ ]:
split = load_split_from_manifest()

print(f"X_train: {split.X_train.shape}")
print(f"X_val:   {split.X_val.shape}")
print(f"X_test:  {split.X_test.shape}")
print(f"\nNumeric features ({len(NUMERIC_FEATURE_COLS)}): {NUMERIC_FEATURE_COLS}")
print(f"Categorical features ({len(CATEGORICAL_FEATURE_COLS)}): {CATEGORICAL_FEATURE_COLS}")

## 2. Build Preprocessor

In [ ]:
preprocessor = build_preprocessor()
preprocessor

## 3. Fit on Training Data Only

In [ ]:
preprocessor, X_train_transformed = fit_transform_train(preprocessor, split.X_train)
print(f"Fitted on X_train: {split.X_train.shape}")
print(f"Transformed X_train shape: {X_train_transformed.shape}")

## 4. Transform Validation and Test (transform only)

In [ ]:
transformed = transform_splits(preprocessor, split)
validation = validate_transformed_splits(split, transformed)

shape_summary = pd.DataFrame(
    {
        "Partition": ["Train", "Validation", "Test"],
        "Original shape": [split.X_train.shape, split.X_val.shape, split.X_test.shape],
        "Transformed shape": [
            transformed.X_train.shape,
            transformed.X_val.shape,
            transformed.X_test.shape,
        ],
    }
)
display(shape_summary)
print("\nValidation checks:")
validation

## 5. Transformed Feature Names

In [ ]:
feature_names = transformed.feature_names
print(f"Original feature count: {ORIGINAL_FEATURE_COUNT}")
print(f"Transformed feature count: {len(feature_names)}")

feature_name_df = pd.DataFrame({"Transformed Feature": feature_names})
feature_name_df

## 6. Numerical Safety Checks

In [ ]:
def numeric_safety_report(name: str, arr: np.ndarray) -> dict:
    return {
        "Partition": name,
        "Has NaN": bool(np.isnan(arr).any()),
        "Has Inf": bool(np.isinf(arr).any()),
        "Min": float(np.min(arr)),
        "Max": float(np.max(arr)),
    }


safety_df = pd.DataFrame(
    [
        numeric_safety_report("Train", transformed.X_train),
        numeric_safety_report("Validation", transformed.X_val),
        numeric_safety_report("Test", transformed.X_test),
    ]
)
safety_df

## 7. Unseen Category Robustness

In [ ]:
probe = split.X_val.iloc[[0]].copy()
probe.loc[probe.index[0], "gender"] = "__UNSEEN_CATEGORY__"
probe_out = preprocessor.transform(probe)

print(f"Probe transform shape: {probe_out.shape}")
print(f"Unseen category handled without error: {validation['unseen_category_transform_ok']}")

## 8. End-to-End Pipeline Helper

In [ ]:
transformed_e2e, fitted_preprocessor, validation_e2e = run_preprocessing_pipeline(split)
print(f"Pipeline validation passed: {validation_e2e['passed']}")
print("Transformed arrays are kept in memory only — not saved to disk in this step.")

## Preprocessing Summary

- **Numeric (4):** `SeniorCitizen`, `tenure`, `MonthlyCharges`, `TotalCharges` → median imputer (defensive) + `StandardScaler`
- **Categorical (15):** one-hot encoded with `handle_unknown='ignore'`
- **Original → transformed features:** 19 → 45
- **Fit policy:** `fit` / `fit_transform` on **`X_train` only**; `X_val` and `X_test` use **`transform` only**
- **`customerID` and `Churn`:** excluded from preprocessor input (handled in separation/split steps)
- **Validation:** row counts preserved; no NaN/Inf; consistent feature count across splits

**Next step (not performed here):** Dummy baseline and model training using this preprocessor inside sklearn `Pipeline`.